#### Name : Harsh Shrimali
#### Subject: NLP - Practical
#### Roll No : 20
#### Course : MSc(AIML) - 3

In [9]:
import re
import string
import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import movie_reviews, gutenberg, stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# NLTK datasets download
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('gutenberg')
nltk.download('movie_reviews')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**1) Perform the relevant text preprocessing steps on the dataset provided (use csv, library and text files all) and write your observations/reasons about the preprocessing you did.**

In [10]:
fileid = movie_reviews.fileids()[0]
raw_library = movie_reviews.raw(fileid)

with movie_reviews.open(fileid) as f:
    raw_txt = f.read()

docs = [(movie_reviews.raw(f), c) for c in movie_reviews.categories() for f in movie_reviews.fileids(c)]
df = pd.DataFrame(docs, columns=['review', 'label'])
df.to_csv('movie_reviews.csv', index=False)
df = pd.read_csv('movie_reviews.csv')

def lower(text):
    return text.lower()

def remove_specials(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

def tokenize(text):
    return word_tokenize(text)

def remove_stopwords(tokens):
    stops = set(stopwords.words('english'))
    return [w for w in tokens if w not in stops and len(w) > 1]

def stemming(tokens):
    ps = PorterStemmer()
    return [ps.stem(w) for w in tokens]

def lemmatizing(tokens):
    wnl = WordNetLemmatizer()
    return [wnl.lemmatize(w) for w in tokens]

def preprocess(text):
    text = lower(text)
    text = remove_specials(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatizing(tokens)
    return ' '.join(tokens)

df_sample = df.sample(5, random_state=42).copy()
df_sample['cleaned_review'] = df_sample['review'].apply(preprocess)
display(df_sample[['label', 'review', 'cleaned_review']])

print("Raw TXT:\n", raw_txt[:150])
print("\nCleaned TXT:\n", preprocess(raw_txt)[:150])

print("\nRaw Library:\n", raw_library[:150])
print("\nCleaned Library:\n", preprocess(raw_library)[:150])

,label,review,cleaned_review
1860,pos,the verdict : spine-chilling drama from horror...,verdict spinechilling drama horror maestro ste...
353,neg,""" the 44 caliber killer has struck again . "" ...",44 caliber killer struck starring john leguiza...
1333,pos,in the company of men made a splash at the sun...,company men made splash sundance film festival...
905,neg,"in the year 2029 , captain leo davidson ( mark...",year 2029 captain leo davidson mark wahlberg b...
1289,pos,[note that followups are directed to rec . art...,note followup directed rec art movie currentfi...


Raw TXT:
 plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continue

Cleaned TXT:
 plot two teen couple go church party drink drive get accident one guy dy girlfriend continues see life nightmare whats deal watch movie sorta find cri

Raw Library:
 plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continue

Cleaned Library:
 plot two teen couple go church party drink drive get accident one guy dy girlfriend continues see life nightmare whats deal watch movie sorta find cri


### Observations / Reasons:
- Lowercasing treats words with different casings as the same word.
- Removing URLs, punctuation, and special characters removes non-informative symbols.
- Tokenization splits raw text into individual words.
- Stopword removal removes very common words (e.g. 'is', 'the', 'at') that don't add meaning.
- Lemmatization reduces words to their root dictionary form (e.g. 'running' -> 'run').

**2) Implement a lemmatizer using the WordNet and compare its output with stemming. Use the Gutenberg Corpus.**

In [11]:
words = gutenberg.words('shakespeare-macbeth.txt')

wnl = WordNetLemmatizer()
ps = PorterStemmer()

sample_words = ['running', 'flies', 'better', 'women', 'corpora', 'leaves', 'generously', 'witches', 'troubled', 'murdered']

res = []
for w in sample_words:
    res.append({
        'Original': w,
        'Stemming': ps.stem(w),
        'Lemmatizer (Noun)': wnl.lemmatize(w, pos='n'),
        'Lemmatizer (Verb)': wnl.lemmatize(w, pos='v')
    })

df_comp = pd.DataFrame(res)
display(df_comp)

clean_tokens = [w.lower() for w in words if w.isalpha() and len(w) > 3][:1000]
print("Unique Raw Tokens:", len(set(clean_tokens)))
print("Unique after Stemming:", len(set([ps.stem(w) for w in clean_tokens])))
print("Unique after Lemmatizing:", len(set([wnl.lemmatize(w) for w in clean_tokens])))

,Original,Stemming,Lemmatizer (Noun),Lemmatizer (Verb)
0,running,run,running,run
1,flies,fli,fly,fly
2,better,better,better,better
3,women,women,woman,women
4,corpora,corpora,corpus,corpora
5,leaves,leav,leaf,leave
6,generously,gener,generously,generously
7,witches,witch,witch,witch
8,troubled,troubl,troubled,trouble
9,murdered,murder,murdered,murder


Unique Raw Tokens: 586
Unique after Stemming: 550
Unique after Lemmatizing: 570


**3) Implement the Bag of Words model to convert a collection of text documents into numerical feature vectors. Use the 20 Newsgroups dataset.**

In [14]:
from sklearn.datasets import fetch_20newsgroups

# Load dataset
cats = ['comp.graphics', 'sci.space', 'rec.sport.baseball']
ng = fetch_20newsgroups(subset='train', categories=cats, remove=('headers', 'footers', 'quotes'))
docs = ng.data[:200]
labels = [ng.target_names[i] for i in ng.target[:200]]

# --- 1. BoW from Scratch ---
stops = set(stopwords.words('english'))
def clean_tokens(text):
    words = re.findall(r'\b[a-z]{3,}\b', text.lower())
    return [w for w in words if w not in stops]
tokenized = [clean_tokens(d) for d in docs]

# Top 20 words for vocab
top_words = Counter(w for d in tokenized for w in d).most_common(20)
vocab = [w for w, _ in top_words]
word2idx = {w: i for i, w in enumerate(vocab)}

# Count word frequencies
bow_scratch = np.zeros((len(docs), len(vocab)), dtype=int)
for i, d in enumerate(tokenized):
    for w in d:
        if w in word2idx:
            bow_scratch[i, word2idx[w]] += 1
df_scratch = pd.DataFrame(bow_scratch, columns=vocab)
df_scratch['category'] = labels
print("BoW From Scratch:")
display(df_scratch.head(10))


# --- 2. BoW using Library ---
vec = CountVectorizer(stop_words='english', max_features=20)
bow_lib = vec.fit_transform(docs)
df_lib = pd.DataFrame(bow_lib.toarray(), columns=vec.get_feature_names_out())
df_lib['category'] = labels
print("BoW using CountVectorizer:")
display(df_lib.head(10))


BoW From Scratch:


,space,launch,year,would,also,satellite,like,one,could,data,...,new,think,program,well,know,first,last,market,get,category
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,comp.graphics
1,0,0,0,0,1,0,1,2,0,0,...,0,0,0,0,0,0,0,0,0,comp.graphics
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,rec.sport.baseball
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sci.space
4,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,rec.sport.baseball
5,0,0,3,1,0,0,0,0,0,0,...,0,2,0,0,0,0,1,0,0,rec.sport.baseball
6,0,0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,comp.graphics
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,comp.graphics
8,0,0,0,1,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,comp.graphics
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,rec.sport.baseball


BoW using CountVectorizer:


,000,02,03,1993,commercial,data,does,don,just,know,...,like,market,new,point,program,satellite,space,think,year,category
0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,comp.graphics
1,0,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,comp.graphics
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,rec.sport.baseball
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sci.space
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,rec.sport.baseball
5,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,2,3,rec.sport.baseball
6,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,comp.graphics
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,comp.graphics
8,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,comp.graphics
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,rec.sport.baseball


**4) Implement TFIDF, BOW and One hot Encoding vectorization from scratch using numpy only.**

In [15]:
# movie_reviews 
corpus = [movie_reviews.raw(f)[:300] for f in movie_reviews.fileids()[:4]]

stops = set(stopwords.words('english'))

def tokenize_doc(doc):
    words = re.findall(r'\b[a-z]{3,}\b', doc.lower())
    return [w for w in words if w not in stops]

def build_vocab(corpus):
    vocab = sorted(list(set(w for d in corpus for w in tokenize_doc(d))))
    word2idx = {w: i for i, w in enumerate(vocab)}
    return vocab, word2idx

vocab, word2idx = build_vocab(corpus)

def bow_from_scratch(corpus, word2idx):
    mat = np.zeros((len(corpus), len(word2idx)), dtype=int)
    for i, d in enumerate(corpus):
        for w in tokenize_doc(d):
            if w in word2idx:
                mat[i, word2idx[w]] += 1
    return mat

def one_hot_from_scratch(corpus, word2idx):
    bow = bow_from_scratch(corpus, word2idx)
    return np.where(bow > 0, 1, 0)

def tfidf_from_scratch(corpus, word2idx):
    bow = bow_from_scratch(corpus, word2idx).astype(float)
    N = len(corpus)
    df = np.sum(bow > 0, axis=0)
    idf = np.log((1 + N) / (1 + df)) + 1
    tfidf = bow * idf
    norm = np.linalg.norm(tfidf, axis=1, keepdims=True)
    norm[norm == 0] = 1.0
    return tfidf / norm

bow_mat = bow_from_scratch(corpus, word2idx)
one_hot_mat = one_hot_from_scratch(corpus, word2idx)
tfidf_mat = tfidf_from_scratch(corpus, word2idx)

print("One-Hot Encoding:")
display(pd.DataFrame(one_hot_mat, columns=vocab))

print("Bag of Words:")
display(pd.DataFrame(bow_mat, columns=vocab))

print("TF-IDF:")
display(pd.DataFrame(np.round(tfidf_mat, 4), columns=vocab))


One-Hot Encoding:


,accident,across,animated,another,attempt,baldwin,based,bastard,bros,brother,...,time,timex,tugboat,two,undercover,viewer,warner,watch,william,worried
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
1,0,1,0,1,0,1,0,1,0,1,...,1,0,1,0,0,0,0,0,1,0
2,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,1,1,0,1,0,0
3,0,0,1,0,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1


Bag of Words:


,accident,across,animated,another,attempt,baldwin,based,bastard,bros,brother,...,time,timex,tugboat,two,undercover,viewer,warner,watch,william,worried
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
1,0,1,0,1,0,1,0,1,0,1,...,1,0,1,0,0,0,0,0,1,0
2,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,1,1,0,1,0,0
3,0,0,1,0,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1


TF-IDF:


,accident,across,animated,another,attempt,baldwin,based,bastard,bros,brother,...,time,timex,tugboat,two,undercover,viewer,warner,watch,william,worried
0,0.1952,0.0000,0.000,0.0000,0.000,0.0000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.1952,0.0000,0.0000,0.000,0.1539,0.0000,0.000
1,0.0000,0.1807,0.000,0.1807,0.000,0.1807,0.0000,0.1807,0.000,0.1807,...,0.1807,0.0000,0.1807,0.0000,0.0000,0.0000,0.000,0.0000,0.1807,0.000
2,0.0000,0.0000,0.000,0.0000,0.000,0.0000,0.1923,0.0000,0.000,0.0000,...,0.0000,0.1923,0.0000,0.0000,0.1923,0.1923,0.000,0.1516,0.0000,0.000
3,0.0000,0.0000,0.189,0.0000,0.189,0.0000,0.0000,0.0000,0.189,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.189,0.0000,0.0000,0.189


**5) Implement TF-IDF vectorization and apply it to a text corpus to extract important features. Use the IMDB movie reviews dataset.**

In [16]:
import tensorflow as tf
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

vocab_size = 5000
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
 
word_index = tf.keras.datasets.imdb.get_word_index()
inv_word_index = {i + 3: w for w, i in word_index.items()}
inv_word_index[0] = "<PAD>"
inv_word_index[1] = "<START>"
inv_word_index[2] = "<UNK>"

texts = []
for seq in x_train[:1000]:
    words = [inv_word_index.get(i, "") for i in seq if i > 3]
    texts.append(" ".join(words))

tokenized_docs = [t.split() for t in texts]

all_words = Counter(w for doc in tokenized_docs for w in doc)
vocab = [w for w, _ in all_words.most_common(2000)]
word2idx = {w: i for i, w in enumerate(vocab)}

# Compute TF
N = len(tokenized_docs)
V = len(vocab)
tf_matrix = np.zeros((N, V))

for i, doc in enumerate(tokenized_docs):
    counts = Counter(doc)
    total = max(len(doc), 1)
    for w, count in counts.items():
        if w in word2idx:
            tf_matrix[i, word2idx[w]] = count / total

# Compute IDF
df_counts = np.sum(tf_matrix > 0, axis=0)
idf = np.log((1 + N) / (1 + df_counts)) + 1
tfidf_scratch = tf_matrix * idf
norms = np.linalg.norm(tfidf_scratch, axis=1, keepdims=True)
norms[norms == 0] = 1.0
tfidf_scratch = tfidf_scratch / norms
mean_scratch = np.mean(tfidf_scratch, axis=0)
top_scratch_idx = np.argsort(mean_scratch)[::-1][:20]

df_scratch = pd.DataFrame({
    'Feature (Scratch)': [vocab[i] for i in top_scratch_idx],
    'Mean TF-IDF': np.round(mean_scratch[top_scratch_idx], 4)
})
print("=== Top Important Features (Scratch) ===")
display(df_scratch)

# TF-IDF & Feature Extraction using Library

tfidf = TfidfVectorizer(max_features=2000, stop_words='english')
X = tfidf.fit_transform(texts)
features = np.array(tfidf.get_feature_names_out())

mean_scores = np.asarray(X.mean(axis=0)).ravel()
top_idx = np.argsort(mean_scores)[::-1][:20]

df_lib = pd.DataFrame({
    'Feature (Library)': features[top_idx],
    'Mean TF-IDF': np.round(mean_scores[top_idx], 4)
})
print("\n=== Top Important Features (TfidfVectorizer) ===")
display(df_lib)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
=== Top Important Features (Scratch) ===


,Feature (Scratch),Mean TF-IDF
0,the,0.2370
1,and,0.1198
2,a,0.1188
3,br,0.1058
4,of,0.1050
5,to,0.1009
6,is,0.0843
7,i,0.0779
8,in,0.0736
9,it,0.0709



=== Top Important Features (TfidfVectorizer) ===


,Feature (Library),Mean TF-IDF
0,br,0.1348
1,movie,0.0687
2,film,0.0597
3,like,0.0344
4,just,0.0307
5,good,0.0298
6,really,0.0270
7,story,0.0267
8,time,0.0252
9,bad,0.0242


**6) Write a program for sentimental analysis (positive and negative) compare any 3 ML models. Do all the steps of preprocessing which you find relevant to task.**

In [17]:
raw_texts = [movie_reviews.raw(f) for f in movie_reviews.fileids()]
labels = [1 if f.startswith('pos') else 0 for f in movie_reviews.fileids()]

def clean_for_sentiment(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    stops = set(stopwords.words('english')) - {'not', 'no', 'never'}
    tokens = [w for w in tokens if w not in stops and len(w) > 2]
    wnl = WordNetLemmatizer()
    return ' '.join([wnl.lemmatize(w) for w in tokens])

cleaned_texts = [clean_for_sentiment(t) for t in raw_texts]

X_tr, X_te, y_tr, y_te = train_test_split(cleaned_texts, labels, test_size=0.2, random_state=42)

vec = TfidfVectorizer(max_features=3000)
X_tr_vec = vec.fit_transform(X_tr)
X_te_vec = vec.transform(X_te)

models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=500),
    'Linear SVC': LinearSVC()
}

res = []
for name, m in models.items():
    m.fit(X_tr_vec, y_tr)
    pred = m.predict(X_te_vec)
    acc = accuracy_score(y_te, pred)
    res.append({'Model': name, 'Accuracy': round(acc * 100, 2)})
    print(f"=== {name} ===")
    print(classification_report(y_te, pred))

display(pd.DataFrame(res))

=== Multinomial Naive Bayes ===
              precision    recall  f1-score   support

           0       0.78      0.81      0.80       199
           1       0.81      0.77      0.79       201

    accuracy                           0.79       400
   macro avg       0.79      0.79      0.79       400
weighted avg       0.79      0.79      0.79       400

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.83      0.81      0.82       199
           1       0.82      0.84      0.83       201

    accuracy                           0.82       400
   macro avg       0.83      0.82      0.82       400
weighted avg       0.83      0.82      0.82       400

=== Linear SVC ===
              precision    recall  f1-score   support

           0       0.85      0.81      0.83       199
           1       0.82      0.86      0.84       201

    accuracy                           0.83       400
   macro avg       0.84      0.83      0.83      

,Model,Accuracy
0,Multinomial Naive Bayes,79.25
1,Logistic Regression,82.50
2,Linear SVC,83.50


**7) Write a program to find edit distance between any two string, output should be in pandas data frame.**

In [ ]:
def edit_distance(s1, s2):
    m = len(s1)
    n = len(s2)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
        
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 2
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
            
    rows = ['#'] + list(s1)
    cols = ['#'] + list(s2)
    df_dp = pd.DataFrame(dp, index=rows, columns=cols)
    return dp[m][n], df_dp

dist, df_table = edit_distance("INTENTION", "EXECUTION")
print("Edit Distance:", dist)
display(df_table)

Edit Distance: 8


,#,E,X,E,C,U,T,I,O,N
#,0,1,2,3,4,5,6,7,8,9
I,1,2,3,4,5,6,7,6,7,8
N,2,3,4,5,6,7,8,7,8,7
T,3,4,5,6,7,8,7,8,9,8
E,4,3,4,5,6,7,8,9,10,9
N,5,4,5,6,7,8,9,10,11,10
T,6,5,6,7,8,9,8,9,10,11
I,7,6,7,8,9,10,9,8,9,10
O,8,7,8,9,10,11,10,9,8,9
N,9,8,9,10,11,12,11,10,9,8


**8) Write a program for n gram language modelling from scratch using numpy and other relevant modulesto predict next word, compare results of unigram, bigram, trigram using perplexity. Store all the n-gram probability in dictionary and then use it for inference.**

In [19]:
raw_sents = gutenberg.sents('shakespeare-macbeth.txt')
clean_sents = [[w.lower() for w in s if w.isalpha()] for s in raw_sents if len(s) > 3]

split = int(0.8 * len(clean_sents))
train_sents = clean_sents[:split]
test_sents = clean_sents[split:]

def train_ngram(sents, n=2):
    ngram_counts = defaultdict(int)
    context_counts = defaultdict(int)
    vocab = set()
    
    for tokens in sents:
        padded = ['<s>'] * (n - 1) + tokens + ['</s>']
        vocab.update(tokens)
        for i in range(len(padded) - n + 1):
            ngram = tuple(padded[i : i + n])
            context = tuple(padded[i : i + n - 1])
            ngram_counts[ngram] += 1
            context_counts[context] += 1
            
    vocab.add('</s>')
    V = len(vocab)
    
    probs = {}
    if n == 1:
        total = sum(ngram_counts.values())
        for ng, c in ngram_counts.items():
            probs[ng] = (c + 1) / (total + V)
    else:
        for ng, c in ngram_counts.items():
            ctx = ng[:-1]
            probs[ng] = (c + 1) / (context_counts[ctx] + V)
            
    return probs, vocab, context_counts

def get_prob(word, context, probs, ctx_counts, V, n):
    ctx = tuple(context)
    ng = ctx + (word,)
    if n == 1:
        return probs.get((word,), 1.0 / (sum(ctx_counts.values()) + V))
    if ng in probs:
        return probs[ng]
    return 1.0 / (ctx_counts.get(ctx, 0) + V)

def compute_perplexity(test_sents, probs, ctx_counts, vocab, n):
    total_log_prob = 0
    W = 0
    V = len(vocab)
    for tokens in test_sents:
        padded = ['<s>'] * (n - 1) + tokens + ['</s>']
        for i in range(len(padded) - n + 1):
            ng = tuple(padded[i : i + n])
            word = ng[-1]
            ctx = ng[:-1]
            p = get_prob(word, ctx, probs, ctx_counts, V, n)
            total_log_prob += np.log(p)
            W += 1
    return np.exp(- total_log_prob / W)

def predict_next(context, probs, ctx_counts, vocab, n, top_k=3):
    ctx = context[-(n - 1):] if n > 1 else []
    if len(ctx) < n - 1:
        ctx = ['<s>'] * (n - 1 - len(ctx)) + ctx
    V = len(vocab)
    cand = []
    for w in vocab:
        if w in ['<s>']:
            continue
        p = get_prob(w, ctx, probs, ctx_counts, V, n)
        cand.append((w, p))
    cand.sort(key=lambda x: x[1], reverse=True)
    return cand[:top_k]

p1, v1, c1 = train_ngram(train_sents, n=1)
p2, v2, c2 = train_ngram(train_sents, n=2)
p3, v3, c3 = train_ngram(train_sents, n=3)

pp1 = compute_perplexity(test_sents, p1, c1, v1, n=1)
pp2 = compute_perplexity(test_sents, p2, c2, v2, n=2)
pp3 = compute_perplexity(test_sents, p3, c3, v3, n=3)

df_ngram = pd.DataFrame({
    'Model': ['Unigram', 'Bigram', 'Trigram'],
    'Perplexity': [round(pp1, 2), round(pp2, 2), round(pp3, 2)]
})
display(df_ngram)

print("Predictions for context ['king'] using bigram:", predict_next(['king'], p2, c2, v2, n=2))
print("Predictions for context ['thou', 'art'] using trigram:", predict_next(['thou', 'art'], p3, c3, v3, n=3))

,Model,Perplexity
0,Unigram,666.36
1,Bigram,1852.01
2,Trigram,2570.08


Predictions for context ['king'] using bigram: [('</s>', 0.0009940357852882703), ('that', 0.0009940357852882703), ('i', 0.0009940357852882703)]
Predictions for context ['thou', 'art'] using trigram: [('the', 0.0010023387905111927), ('so', 0.0006682258603407952), ('too', 0.0006682258603407952)]


**9) Implement Naïve Bayes Classifier for Text classification use dataset of your choice from nltk corpus, preprocess it and use it for classification. Deal all the cases such as out of vocabulary, smoothing etc. Compare the results with library function of naïve bayes using relevant metrics**

In [20]:
docs = []
labels = []
for f in movie_reviews.fileids():
    words = [w.lower() for w in movie_reviews.words(f) if w.isalpha() and len(w) > 2]
    docs.append(words)
    labels.append(f.split('/')[0])

X_tr_tok, X_te_tok, y_tr, y_te = train_test_split(docs, labels, test_size=0.25, random_state=42)

def fit_nb(docs, labels, alpha=1.0):
    classes = list(set(labels))
    priors = {c: labels.count(c) / len(labels) for c in classes}
    word_counts = {c: defaultdict(int) for c in classes}
    total_words = {c: 0 for c in classes}
    vocab = set()
    
    for tokens, l in zip(docs, labels):
        for w in tokens:
            vocab.add(w)
            word_counts[l][w] += 1
            total_words[l] += 1
            
    return priors, word_counts, total_words, vocab, classes

def predict_doc(tokens, priors, word_counts, total_words, vocab, classes, alpha=1.0):
    V = len(vocab)
    posteriors = {}
    for c in classes:
        log_prob = math.log(priors[c])
        denom = total_words[c] + alpha * V
        for w in tokens:
            if w in vocab:
                log_prob += math.log((word_counts[c][w] + alpha) / denom)
        posteriors[c] = log_prob
    return max(posteriors, key=posteriors.get)

def predict_nb(docs, priors, word_counts, total_words, vocab, classes, alpha=1.0):
    return [predict_doc(d, priors, word_counts, total_words, vocab, classes, alpha) for d in docs]

priors, word_counts, total_words, vocab, classes = fit_nb(X_tr_tok, y_tr)
y_pred_scratch = predict_nb(X_te_tok, priors, word_counts, total_words, vocab, classes)

# Library Naive Bayes
tr_str = [' '.join(d) for d in X_tr_tok]
te_str = [' '.join(d) for d in X_te_tok]

vec = CountVectorizer()
X_tr_vec = vec.fit_transform(tr_str)
X_te_vec = vec.transform(te_str)

clf = MultinomialNB()
clf.fit(X_tr_vec, y_tr)
y_pred_lib = clf.predict(X_te_vec)

df_comp = pd.DataFrame([
    {'Classifier': 'Scratch Naïve Bayes', 'Accuracy': round(accuracy_score(y_te, y_pred_scratch) * 100, 2)},
    {'Classifier': 'Library MultinomialNB', 'Accuracy': round(accuracy_score(y_te, y_pred_lib) * 100, 2)}
])
display(df_comp)

print("Classification Report (Scratch):\n", classification_report(y_te, y_pred_scratch))
print("Classification Report (Library):\n", classification_report(y_te, y_pred_lib))

,Classifier,Accuracy
0,Scratch Naïve Bayes,80.0
1,Library MultinomialNB,80.0


Classification Report (Scratch):
               precision    recall  f1-score   support

         neg       0.79      0.82      0.81       257
         pos       0.81      0.77      0.79       243

    accuracy                           0.80       500
   macro avg       0.80      0.80      0.80       500
weighted avg       0.80      0.80      0.80       500

Classification Report (Library):
               precision    recall  f1-score   support

         neg       0.79      0.82      0.81       257
         pos       0.81      0.77      0.79       243

    accuracy                           0.80       500
   macro avg       0.80      0.80      0.80       500
weighted avg       0.80      0.80      0.80       500

